# Notebook 03: Detection Loss Functions

**Module:** 08 Object Detection  
**Duration:** ~2 hrs

---

## Learning Objectives

1. Decompose detection loss into classification + localization
2. Derive Smooth L1 (Huber) loss for box regression
3. Understand focal loss for detection
4. Match losses to one-stage vs two-stage detectors

## Detection Loss

$$L = L_{\text{cls}} + \lambda L_{\text{loc}}$$

**Classification:** Cross-entropy (or focal loss) on object class + background.

**Localization:** Smooth L1 between predicted and target box offsets.

**Smooth L1:**
$$L(x) = \begin{cases} 0.5 x^2 & |x| < 1 \\ |x| - 0.5 & \text{otherwise} \end{cases}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
import torch.nn.functional as F

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
rng = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
def smooth_l1(pred, target, beta=1.0):
    diff = (pred - target).abs()
    return torch.where(diff < beta, 0.5 * diff ** 2 / beta, diff - 0.5 * beta).mean()

pred_box = torch.tensor([0.1, -0.2, 0.05, 0.3])
target_box = torch.tensor([0.0, 0.0, 0.0, 0.0])
print(f'Smooth L1: {smooth_l1(pred_box, target_box):.4f}')

# Focal loss for classification (RetinaNet)
def focal_loss_cls(logits, targets, gamma=2.0, alpha=0.25):
    ce = F.cross_entropy(logits, targets, reduction='none')
    p = torch.exp(-ce)
    return (alpha * (1 - p) ** gamma * ce).mean()

logits = torch.randn(8, 5)
targets = torch.randint(0, 5, (8,))
print(f'Focal cls loss: {focal_loss_cls(logits, targets):.4f}')

## Anchor Matching

Assign each anchor/prior box to GT:
- IoU > 0.7 → positive
- IoU < 0.3 → negative (background)
- Between → ignore

Regression targets are **offsets** from anchor to GT box (Faster R-CNN parameterization).

---

## Summary

Detection = cls loss + loc loss; focal loss handles foreground/background imbalance.

**Next:** [04_R_CNN.ipynb](04_R_CNN.ipynb)